# Alert 생성기 실험 ① — 모델이 고른 거래: 묶기 전 vs 묶은 뒤

이진 모델이 의심이라고 한 거래가 Alert 생성기를 거치면서 **무엇이 떨어지고 무엇이 들어오는지**를 정답 라벨로 잰다.

| 단계 | 고른 거래 | 서비스 코드 |
|---|---|---|
| ① 모델 씨앗 | 점수 ≥ 의심 기준 | `alert_pipeline.save_alerts` 의 씨앗 수집 |
| ② 씨앗 보강 | ① + 씨앗과 이어진 낮은 점수 거래 | `alerts/seed_promotion.promote` |
| ③ 묶은 뒤 | Alert 에 들어간 거래 (씨앗 + 연결 거래) | `alerts/block_graph.build_block_graphs` |

- 생성기는 서비스 저장소(`money_laundry`, dev)의 코드를 그대로 불러온다. 복사본이 아니다.
- **서비스와 다른 점**: 서비스는 매일 이어서 돌고 이미 공개된 Alert 를 고정한다. 여기서는 평가 구간의 씨앗을 한 번에 넣고 한 번 돌린다.
- 평가 구간 앞 `PAD_DAYS` 일의 거래는 생성기가 구조를 볼 때만 쓴다. 씨앗과 지표는 평가 구간 안의 거래만이다.
- 처음에는 `DATASET = 'HI-Small'` 로 끝까지 한 번 돌려 본다.

## 0. 설정

In [ ]:
DATASET = 'HI-Large'             # 처음에는 'HI-Small'
SCORES = None                    # 이진 모델 테스트 점수 parquet (edge_id, y, score)
                                 # 예: '../binary_classification/results/HI-Large/<실험명>/<모델>_test_scores.parquet'
THRESHOLD = 0.194997844          # 의심 기준. 서비스 이진 모델(multi_fraudgt_hgb) spec 의 RECOMMENDED_THRESHOLD
SEED_LOW, CHAIN_FLOOR = 0.1, 0.01   # 씨앗 보강 값(같은 spec 의 ALERT_SEED_LOW, ALERT_CHAIN_FLOOR). 보강을 끄려면 None, None
PAD_DAYS = 14                    # 평가 구간 앞으로 더 읽는 날짜 수. 생성기의 블록 기간 상한(336시간)과 같다
BUDGET_SCALE = 1                 # 생성기 탐색 예산 배수. 5절에서 예산 초과가 나면 올린다

`THRESHOLD`, `SEED_LOW`, `CHAIN_FLOOR` 는 **서비스 이진 모델의 점수 척도**에 맞춘 값이다.
다른 모델의 점수 파일을 넣으면 그 모델의 기준으로 바꾸고, 보강 값이 없으면 `None, None` 으로 둔다.

In [ ]:
from pathlib import Path
import json, subprocess, time
import duckdb, numpy as np, pandas as pd
import gen_lib as GL

# 평가 구간 = 이진분류 실험의 테스트 구간 (sampling_lib.TEST, bc_lib.WINDOWS 와 같은 값)
TEST_WINDOW = {'HI-Large': ('2022-10-17', '2022-10-30'), 'HI-Small': ('2022-09-09', '2022-09-10')}[DATASET]
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'parquets').is_dir() or (p / 'datasets').is_dir()), None)
assert ROOT is not None, 'parquets/ 폴더를 못 찾음. 데이터가 있는 GPU 서버의 저장소 안에서 열어야 한다'
BASE = ROOT / 'parquets' / DATASET
OUT = Path.cwd() / 'results' / DATASET / 'before_after_grouping'
OUT.mkdir(parents=True, exist_ok=True)
assert SCORES is not None, '0절의 SCORES 에 점수 파일 경로를 넣는다'
print(BASE, '| 평가 구간', TEST_WINDOW)

## 1. 생성기 불러오기

In [ ]:
WORKER = GL.find_worker()
G = GL.load_generator(WORKER)
POLICY = G.BlockPolicy()
print(WORKER)
print('정책', POLICY.version, '| 블록 기간 상한', POLICY.span_max_us // 3_600_000_000, '시간 | 허브 기준', POLICY.hub_min_activity, '건')
# 어느 커밋의 생성기로 돌렸는지 남긴다
COMMIT = subprocess.run(['git', '-C', str(WORKER), 'log', '-1', '--format=%h %ad %s', '--date=short'], capture_output=True, text=True).stdout.strip()
print(COMMIT)

## 2. 거래 읽기
평가 구간과 그 앞 `PAD_DAYS` 일을 시간순으로 읽는다.

In [ ]:
con = duckdb.connect()
a, b = TEST_WINDOW
start = (pd.Timestamp(a) - pd.Timedelta(days=PAD_DAYS)).date()
t = time.perf_counter()
tx = con.execute(f"""
    SELECT edge_id, ts, src, dst, paid, payment_currency, CAST(is_laundering AS TINYINT) AS y, day
    FROM read_parquet('{BASE / 'transactions.parquet'}')
    WHERE day BETWEEN DATE '{start}' AND DATE '{b}'
    ORDER BY ts, edge_id""").df()
print(f'{len(tx):,} 건  {tx.ts.min()} ~ {tx.ts.max()}  ({time.perf_counter() - t:.0f}s)')
tx.head(3)

### 2-1. USD 환산
생성기는 금액 비율로 흐름을 잇는다. 서비스와 같은 고정 환율로 보낸 금액을 USD 로 바꾼다.

In [ ]:
tx['usd'] = GL.to_usd(tx.paid, tx.payment_currency)
print('환율을 모르는 통화:', list(tx.loc[tx.usd.isna(), 'payment_currency'].unique()))   # 비어 있어야 정상
tx.groupby('payment_currency').agg(n=('usd', 'size'), usd_median=('usd', 'median'))

### 2-2. 패턴 라벨 붙이기
세탁 거래 중 Patterns 파일에 없는 것은 `NO_PATTERN` 으로 둔다.

In [ ]:
pat = con.execute(f"""SELECT edge_id, any_value(pattern_type) AS pattern_type
                      FROM read_parquet('{BASE / 'patterns.parquet'}') WHERE edge_id IS NOT NULL GROUP BY edge_id""").df()
n_before = len(tx)
tx = tx.merge(pat, on='edge_id', how='left')
assert len(tx) == n_before and tx.ts.is_monotonic_increasing        # 행 수와 순서가 그대로여야 한다
tx['pattern_type'] = np.where(tx.y == 1, tx.pattern_type.fillna('NO_PATTERN'), '')
in_eval = ((pd.to_datetime(tx.day) >= pd.Timestamp(a)) & (pd.to_datetime(tx.day) <= pd.Timestamp(b))).to_numpy()
print(f'평가 구간 거래 {in_eval.sum():,} | 그중 세탁 {int(tx.y[in_eval].sum()):,}')
tx[in_eval & (tx.y == 1).to_numpy()].pattern_type.value_counts()

## 3. 모델 점수 → 씨앗 (①)

In [ ]:
scores = pd.read_parquet(SCORES)          # edge_id, y, score
n_before = len(tx)
tx = tx.merge(scores[['edge_id', 'score']].drop_duplicates('edge_id'), on='edge_id', how='left')
assert len(tx) == n_before and tx.ts.is_monotonic_increasing
score = tx.score.to_numpy(dtype='float64')
print(f'점수 {len(scores):,} 건 | 평가 구간인데 점수가 없는 거래 {int((in_eval & np.isnan(score)).sum()):,} (0 이어야 정상)')
# 점수 파일의 라벨과 거래 파일의 라벨이 같은지
check = scores.merge(tx[['edge_id', 'y']], on='edge_id', suffixes=('_score', '_tx'))
print('라벨 불일치:', int((check.y_score != check.y_tx).sum()))

In [ ]:
y = tx.y.to_numpy()
pattern_type = tx.pattern_type.to_numpy()
pattern_mask = (y == 1) & (pattern_type != 'NO_PATTERN')
seed_model = in_eval & (np.nan_to_num(score, nan=-1.0) >= THRESHOLD)
stages = [GL.stage_metrics('① 모델 씨앗', seed_model, y, in_eval, pattern_mask)]
pd.DataFrame(stages)

## 4. 씨앗 보강 (②)
기준 미만이지만 씨앗과 계좌를 공유하거나(점수 ≥ `SEED_LOW`) 금액이 이어지는(점수 ≥ `CHAIN_FLOOR`) 거래를 씨앗에 더한다.

In [ ]:
t = time.perf_counter()
edges = GL.make_edges(G, tx)
print(f'생성기 입력 {len(edges):,} 건 ({time.perf_counter() - t:.0f}s)')

In [ ]:
if SEED_LOW is None or CHAIN_FLOOR is None:
    promoted = np.array([], dtype=np.int64)
else:
    t = time.perf_counter()
    promoted = GL.promote_seeds(G, edges, score, np.flatnonzero(seed_model), SEED_LOW, CHAIN_FLOOR)
    print(f'보강으로 들어온 거래 {len(promoted):,} 건 ({time.perf_counter() - t:.0f}s)')
seed_all = seed_model.copy()
seed_all[promoted] = True
stages.append(GL.stage_metrics('② 씨앗 보강', seed_all, y, in_eval, pattern_mask))
pd.DataFrame(stages)

In [ ]:
# 보강으로 들어온 거래가 실제로 무엇이었나
is_promoted = np.zeros(len(tx), dtype=bool)
is_promoted[promoted] = True
GL.split_by_label(is_promoted, y, pattern_type)

## 5. 생성기 실행
씨앗을 한 번에 넣고 묶는다. HI-Large 는 색인을 만드는 데 몇 분 걸릴 수 있다.

In [ ]:
try:
    batch, info = GL.run_generator(G, edges, np.flatnonzero(seed_all), budget_scale=BUDGET_SCALE)
except G.GraphBudgetExceeded as error:
    raise RuntimeError(f'생성기 탐색 예산 초과: {error}. 0절의 BUDGET_SCALE 을 올려 다시 돌린다') from error
info

In [ ]:
role, graph, reason, alerts = GL.batch_table(batch, len(tx))
print(f'Alert {len(alerts):,} 개 | Alert 에 못 들어간 씨앗 {len(batch.unassigned):,} 건')
batch.stats

## 6. 세 단계 비교 (③)

In [ ]:
member = role > GL.ROLE_NONE
print('Alert 구성 거래 중 평가 구간 밖(앞쪽 PAD 구간) 거래:', int((member & ~in_eval).sum()), '건 → 아래 지표에서는 뺀다')
stages.append(GL.stage_metrics('③ 묶은 뒤', member, y, in_eval, pattern_mask))
result = pd.DataFrame(stages)
result

**읽는 법**
- ① → ③ 에서 `정상` 이 줄어든 만큼이 생성기가 걸러낸 오탐이다.
- ① → ③ 에서 `세탁` 이 줄어든 만큼이 생성기 때문에 놓친 세탁이다.
- `재현율_패턴만` 은 Patterns 파일에 있는 세탁만 분모로 한 값이다. 무패턴 제외 기준으로 볼 때 쓴다.

## 7. 무엇이 떨어지고 무엇이 들어왔나

In [ ]:
# 7-1. Alert 에 못 들어간 씨앗: 탈락 사유 × 실제 라벨
dropped = seed_all & (role == GL.ROLE_NONE)
label = np.where(y == 1, pattern_type, '정상')
pd.crosstab(pd.Series(reason[dropped], name='탈락 사유'), pd.Series(label[dropped], name='실제'), margins=True)

In [ ]:
# 7-2. 씨앗이 아닌데 구조상 Alert 에 들어온 거래(연결 거래)
GL.split_by_label(role == GL.ROLE_CONNECTION, y, pattern_type)

In [ ]:
# 7-3. 패턴별 재현율이 단계마다 어떻게 바뀌나
rows = []
for kind in sorted(set(pattern_type[in_eval & (y == 1)])):
    base = in_eval & (y == 1) & (pattern_type == kind)
    rows.append(dict(패턴=kind, 세탁_거래=int(base.sum()),
                     모델_씨앗=round((base & seed_model).sum() / base.sum(), 4),
                     씨앗_보강=round((base & seed_all).sum() / base.sum(), 4),
                     묶은_뒤=round((base & member).sum() / base.sum(), 4)))
by_pattern = pd.DataFrame(rows)
by_pattern

## 8. Alert 단위로 보기
"맞은 Alert" 의 기준은 아직 정하지 않았다. 세탁 거래가 1건 이상 / 절반 이상 / 전부인 Alert 수를 같이 본다.

In [ ]:
alerts = GL.alert_purity(graph, role, y, alerts)
summary = dict(Alert_수=len(alerts), 세탁_1건_이상=int((alerts.laundering >= 1).sum()),
               세탁_절반_이상=int((alerts.purity >= 0.5).sum()), 전부_세탁=int((alerts.purity == 1).sum()),
               전부_정상=int((alerts.laundering == 0).sum()))
print(summary)
alerts.members.describe(percentiles=[.5, .9, .99]).round(1)

In [ ]:
# 어떤 구조 근거로 묶였나 (STAR_IN, STAR_OUT, RELAY, PASS, PATH, BRIDGE, ...)
alerts.groupby('kinds').agg(n=('alert', 'size'), purity_mean=('purity', 'mean'), members_mean=('members', 'mean')).sort_values('n', ascending=False).head(15).round(3)

## 9. 저장

In [ ]:
result.to_csv(OUT / 'stages.csv', index=False)
by_pattern.to_csv(OUT / 'by_pattern.csv', index=False)
alerts.to_csv(OUT / 'alerts.csv', index=False)
keep = seed_all | member
pd.DataFrame(dict(edge_id=tx.edge_id[keep], y=y[keep], pattern_type=pattern_type[keep], score=score[keep],
                  seed_model=seed_model[keep], promoted=is_promoted[keep], role=role[keep], alert=graph[keep],
                  reason=reason[keep])).to_parquet(OUT / 'members.parquet', index=False)
json.dump(dict(dataset=DATASET, scores=str(SCORES), threshold=THRESHOLD, seed_low=SEED_LOW, chain_floor=CHAIN_FLOOR,
               pad_days=PAD_DAYS, test_window=TEST_WINDOW, generator_commit=COMMIT, info=info, alert_summary=summary),
          open(OUT / 'run.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print(sorted(p.name for p in OUT.iterdir()))